# Decision Tree: Optimised Variants (Depth, Leaf Size, Impurity Measure)

**Project:** Machine-learning-based recognition of production and changeover activities  
**CRISP-DM phase:** Modelling / Evaluation

## Purpose

Notebook 04 trained unrestricted trees. This notebook compares **six decision-tree variants** for `Label_03` (**1 = Production, 0 = Changeover**) to see how restricting the tree changes the result:

- impurity measure: Gini or entropy;
- `max_depth`: unrestricted (`None`), 5 or 3 (a shallower tree is simpler and less likely to memorize);
- `min_samples_leaf`: 1 or 5 (a larger value avoids very small, over-specific branches).

For every variant it reports accuracy, F1, AUC, sensitivity, precision, specificity and Cohen's kappa, plus the training time and the training vs. test accuracy (a large gap signals overfitting). It then shows plots and the confusion matrix of the best variant and saves the results.

## Workflow

Run notebook 01 first; it creates `normalized_data.csv`. The thesis describes this comparison in Section 4.2.

> **Data:** The dataset is public (Zenodo, licence CC BY 4.0) and is **not** included in this repository. See the project README for the download link and citation. This notebook reads `normalized_data.csv`, which notebook 01 creates in the project root. Generated files are written to `../outputs/` and are not committed.

## Important limitations

- **Random split on time-series data.** The split is a stratified *random* split of rows. Neighbouring time steps of a machine are very similar, so near-duplicates can end up in both training and test sets and the scores can be **optimistic**. A chronological or group-wise split is a stricter test.
- **Origin of the 40 features.** The 40 features are the 20 most negative and the 20 most positive Pearson correlations with `Label_03` from notebook 01 (Section 8). That ranking was computed on the whole dataset, including later test rows, so the test scores are slightly biased.
- **Class coding.** Class `1` = Production is the positive class and class `0` = Changeover.
- **The best variant is chosen on the test set.** All six variants are ranked by their test scores, and the best one is then reported. This is optimistic; an unbiased choice needs a separate validation set or cross-validation on the training data.

## 1. Environment and configuration

Install the dependencies with `pip install -r requirements.txt` (run from the project root). All paths are relative to this notebook, which is expected to live in the `notebooks/` folder.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    PrecisionRecallDisplay, RocCurveDisplay, accuracy_score,
    average_precision_score, cohen_kappa_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

DATA_PATH = Path("../normalized_data.csv")   # created by notebook 01
OUTPUT_DIR = Path("../outputs")              # local results, not for upload

LABEL_COL = "Label_03"
SEED = 42
TEST_SIZE = 0.2

## 2. Load the normalized data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Run notebook 01 first (it creates normalized_data.csv in the project root) "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

if LABEL_COL not in df.columns:
    raise KeyError(
        f"Target column {LABEL_COL!r} not found. "
        f"The file has {len(df.columns)} columns."
    )

## 3. Select the 40 features and split the data

Same features, split and seed as in notebooks 02 to 04. Missing and infinite values are replaced by the median of the **training** rows.

In [ ]:
feature_list = [
    'Spindle_smoothed_speed',
    'Program_status',
    'Z-axis_position_WCS',
    'Magazine_9998_number_Place_2',
    'Door_open',
    'Z-axis_position_MCS',
    'Magazine_9998_position_Place_3',
    'Program_Idle',
    'Magazine_position_Place_9',
    'Tool_number_Magazine_Place_43',
    'Magazine_number_Place_6',
    'Magazine_number_Place_7',
    'Magazine_number_Place_5',
    'Magazine_9998_position_Place_1',
    'Tool_number_Magazine_Place_9',
    'Magazine_number_Place_8',
    'Tool_magazine_position_WCS',
    'Tool_magazine_position_MCS',
    'Magazine_position_Place_17',
    'Number_of_tools',
    'Feed_override',
    'Rapid_traverse_button',
    'C_Axis_Motor_temperature',
    'Y_Axis_Motor_temperature',
    'Door_status_locked',
    'Spindle_position_WCS',
    'Spindle_position_MCS',
    'X-axis_position_MCS',
    'Net_running_time_program',
    'X_Axis_motor_temperature',
    'B_Axis_Motor_temperature',
    'Program_line_number',
    'Tool_number_Magazine_9998_Place_1',
    'X-axis_position_WCS',
    'Operation_mode',
    'LED_Chip_flushing',
    'Spindle_angular_velocity',
    'Spindle_speed',
    'Spindle_smoothed_modulation_depth',
    'LED_Coolant',
]

missing_features = [c for c in feature_list if c not in df.columns]
if missing_features:
    print("Warning: these features are missing and will be skipped:", missing_features)

feature_list = [c for c in feature_list if c in df.columns]
if not feature_list:
    raise ValueError("None of the expected features were found in the data.")
print(f"Using {len(feature_list)} features")

In [ ]:
X = df[feature_list].replace([np.inf, -np.inf], np.nan)
y = df[LABEL_COL].astype(int).to_numpy()

if set(np.unique(y)) != {0, 1}:
    raise ValueError(
        f"{LABEL_COL} must contain exactly the classes 0 and 1, "
        f"found: {np.unique(y).tolist()}"
    )

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 4. Train and evaluate the six variants

The table is sorted by AUC, then accuracy, then speed. Model names read `DT_<criterion>_d<max_depth>_leaf<min_samples_leaf>` (`dNone` = unrestricted depth). The training time depends on your computer.

In [ ]:
def eval_metrics(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "ACC": accuracy_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "AUC": roc_auc_score(y_true, proba),
        "Sensitivity": recall_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "Kappa": cohen_kappa_score(y_true, pred),
    }


variations = [
    {"criterion": "gini",    "max_depth": None, "min_samples_leaf": 1},
    {"criterion": "entropy", "max_depth": None, "min_samples_leaf": 1},
    {"criterion": "gini",    "max_depth": 5,    "min_samples_leaf": 1},
    {"criterion": "entropy", "max_depth": 5,    "min_samples_leaf": 1},
    {"criterion": "gini",    "max_depth": 3,    "min_samples_leaf": 5},
    {"criterion": "entropy", "max_depth": 3,    "min_samples_leaf": 5},
]

rows, models = [], {}
for p in variations:
    t0 = time.time()
    model = DecisionTreeClassifier(
        criterion=p["criterion"], max_depth=p["max_depth"],
        min_samples_leaf=p["min_samples_leaf"], class_weight="balanced",
        random_state=SEED,
    ).fit(X_train, y_train)
    seconds = time.time() - t0

    proba = model.predict_proba(X_test)[:, 1]
    name = f"DT_{p['criterion']}_d{p['max_depth']}_leaf{p['min_samples_leaf']}"
    row = eval_metrics(y_test, proba)
    row.update({
        "Model": name,
        "Seconds": seconds,
        "ACC_train": accuracy_score(y_train, model.predict(X_train)),
        "ACC_test": accuracy_score(y_test, model.predict(X_test)),
    })
    rows.append(row)
    models[name] = (model, proba)

results = pd.DataFrame(rows).round(4)
results["Rank_AUC"] = results["AUC"].rank(ascending=False, method="min").astype(int)
results["Rank_ACC"] = results["ACC"].rank(ascending=False, method="min").astype(int)
results["Rank_Speed"] = results["Seconds"].rank(ascending=True, method="min").astype(int)
results = results.sort_values(["Rank_AUC", "Rank_ACC", "Rank_Speed"]).reset_index(drop=True)
display(results)

best_name = results.loc[0, "Model"]
best_model, best_proba = models[best_name]
print("Selected variant (best AUC, then accuracy, on the test set):", best_name)

## 5. Training vs. test accuracy

If the training accuracy is clearly higher than the test accuracy, the tree has memorized the training data. A tree with unrestricted depth reaches a training accuracy of 100 %, because it can keep splitting until every training row is separated.

In [ ]:
order = results["Model"].tolist()
x = np.arange(len(order))

plt.figure(figsize=(6, 4))
plt.plot(x, results["ACC_train"], marker="o", label="train")
plt.plot(x, results["ACC_test"], marker="o", label="test")
plt.xticks(x, order, rotation=45, ha="right")
plt.ylim(0, 1)
plt.ylabel("accuracy")
plt.title("Accuracy per variant")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Compare the variants

Bar chart of accuracy, F1 and AUC per variant, and ROC and precision-recall curves of the three best variants (all curves in one plot).

In [ ]:
width = 0.26
plt.figure(figsize=(8, 4))
plt.bar(x - width, results["ACC"], width, label="ACC", color="#08519c")
plt.bar(x,         results["F1"],  width, label="F1",  color="#6baed6")
plt.bar(x + width, results["AUC"], width, label="AUC", color="#c6dbef")
plt.xticks(x, order, rotation=45, ha="right")
plt.ylim(0, 1.0)
plt.ylabel("score")
plt.title("Model metrics by variant")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

top3 = results["Model"].head(3).tolist()

fig, ax = plt.subplots(figsize=(5, 4))
for name in top3:
    RocCurveDisplay.from_predictions(y_test, models[name][1], name=name, ax=ax)
ax.set_title("ROC: top 3 variants")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
for name in top3:
    PrecisionRecallDisplay.from_predictions(y_test, models[name][1], name=name, ax=ax)
ax.set_title("Precision-Recall: top 3 variants")
plt.tight_layout()
plt.show()

## 7. Best variant: confusion matrix and detailed metrics

In [ ]:
pred_best = (best_proba >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred_best)

plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title(f"Confusion matrix: {best_name}")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
detail = pd.DataFrame([{
    **eval_metrics(y_test, best_proba),
    "AP": average_precision_score(y_test, best_proba),
    "TN": tn, "FP": fp, "FN": fn, "TP": tp,
}], index=[best_name]).round(4)
display(detail)

## 8. Node impurities of the best tree

For every node: the number of training rows, the class-1 share `p1` (class-weighted, because of `class_weight="balanced"`) and the entropy. The sample counts come from `tree_.n_node_samples`, because in scikit-learn 1.4 and newer `tree_.value` holds class fractions instead of counts.

In [ ]:
def entropy(p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))


tree = best_model.tree_
values = tree.value.squeeze(axis=1)
totals = values.sum(axis=1)
p1 = np.divide(values[:, 1], totals, out=np.zeros_like(totals), where=totals > 0)

nodes_df = pd.DataFrame({
    "node_id": np.arange(tree.node_count),
    "samples": tree.n_node_samples,
    "p1": p1,
    "entropy": entropy(p1),
    "is_leaf": tree.children_left == -1,
}).sort_values("samples", ascending=False)

print(f"Nodes in the tree: {len(nodes_df):,} (depth {best_model.get_depth()})")
display(nodes_df.head(10))

## 9. Save the results locally

Written to `../outputs/` (not for upload): the metrics of all variants, the node table of the best tree, and the test predictions of the best variant.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_out = OUTPUT_DIR / "dt_variations_metrics.csv"
nodes_out = OUTPUT_DIR / "dt_nodes_entropy.csv"
preds_out = OUTPUT_DIR / "dt_best_predictions.csv"

results.to_csv(metrics_out, index=False)
nodes_df.to_csv(nodes_out, index=False)
pd.DataFrame({"y_true": y_test, "y_proba": best_proba, "y_pred": pred_best}).to_csv(
    preds_out, index=False
)

for path in (metrics_out, nodes_out, preds_out):
    print("Saved:", path)

## 10. Summary and conclusion

Restricting the tree changes the picture clearly: the unrestricted trees score highest, but they also fit the training data perfectly, while trees limited to depth 5 or 3 score much lower on the test set. Within each depth setting, Gini and entropy behave very similarly.

**Reference results from the author's original run** (the cleaned notebook may give slightly different numbers when re-run; the training times depend on the computer):

| Variant | Accuracy | F1 | AUC | Sensitivity | Precision |
|---|---|---|---|---|---|
| `DT_entropy_dNone_leaf1` | 0.9949 | 0.9916 | 0.9942 | 0.9923 | 0.9908 |
| `DT_gini_dNone_leaf1` | 0.9947 | 0.9912 | 0.9939 | 0.9917 | 0.9908 |
| `DT_entropy_d5_leaf1` | 0.9061 | 0.8613 | 0.9693 | 0.9678 | 0.7759 |
| `DT_gini_d5_leaf1` | 0.9115 | 0.8667 | 0.9543 | 0.9550 | 0.7933 |
| `DT_gini_d3_leaf5` | 0.8165 | 0.7504 | 0.9068 | 0.9154 | 0.6357 |
| `DT_entropy_d3_leaf5` | 0.8081 | 0.7386 | 0.9030 | 0.9001 | 0.6262 |

The best variant is the same unrestricted entropy tree as in notebook 04, so the "optimisation" did not beat the baseline. Because the best variant was chosen on the test set and the split is random, the top scores are optimistic.

**Next steps:** select the variant with cross-validation on the training data, and use a chronological or group-wise split.